# MindMate AI - Emotion model training (XLM-R base)

Ye notebook 5 labels (joy, sadness, fear_anxiety, anger, neutral) par
English + Roman Urdu emotion model train karti hai.

**Steps (sirf ye karna hai):**
1. Runtime -> Change runtime type -> **T4 GPU** -> Save
2. Left side Files icon -> ye 3 files upload karein (ya pehli cell chalne par upload button aayega):
   `emotion_en.jsonl`, `emotion_roman_ur.jsonl`, `stats.json`
   (laptop par: `D:\MindMate FYP Project\mindmate\ml\notebooks\emotion_training\` (isi notebook ke saath wale folder mein))
3. Runtime -> **Run all**
4. Aakhir mein `emotion_model.zip` download hogi. Isay
   `D:\MindMate FYP Project\mindmate\ml\export\` mein extract karein.

Rules jo notebook khud enforce karti hai: frozen test set ka hash check,
thresholds/temperature sirf validation par, results har language ke alag
(kabhi pooled nahi), 95% confidence intervals.

In [ ]:
# 1) Install pinned export libraries (torch + transformers come with Colab;
#    their exact versions are recorded in the report).
!pip install -q onnx==1.23.2 onnxruntime==1.31.1 onnxscript==0.7.2

In [ ]:
# 2) Settings
import os, json, hashlib, random, math, time
from pathlib import Path
from collections import Counter

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DATA_DIR = Path(os.environ.get("MINDMATE_DATA_DIR", "/content"))
OUT_DIR = Path(os.environ.get("MINDMATE_OUT_DIR", "/content/emotion_model"))
BEST_DIR = Path("/content/best_model")

# Order MUST match backend app/emotion/__init__.py LABELS.
LABELS = ["joy", "sadness", "fear_anxiety", "anger", "neutral"]
L2I = {l: i for i, l in enumerate(LABELS)}
LANGS = ["en", "roman_ur"]

MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 128
BATCH = 32
EPOCHS = 3
LR = 2e-5
WARMUP = 0.06
SEED = 42

In [ ]:
# 3) Upload data files if they are not there yet
needed = ["emotion_en.jsonl", "emotion_roman_ur.jsonl", "stats.json"]
missing = [n for n in needed if not (DATA_DIR / n).exists()]
if missing:
    from google.colab import files
    print("Please upload:", missing)
    import re
    for name, blob in files.upload().items():
        clean = re.sub(r" \(\d+\)(?=\.)", "", name)  # "stats (1).json" -> "stats.json"
        (DATA_DIR / clean).write_bytes(blob)
missing = [n for n in needed if not (DATA_DIR / n).exists()]
assert not missing, f"Still missing: {missing}"
print("data files OK")

In [ ]:
# 4) Load data + verify the FROZEN test set has not changed
STATS = json.loads((DATA_DIR / "stats.json").read_text(encoding="utf-8"))
data = {}
for lang in LANGS:
    lines = (DATA_DIR / f"emotion_{lang}.jsonl").read_text(encoding="utf-8").splitlines()
    rows = [json.loads(l) for l in lines if l.strip()]
    test_bytes = "".join(
        json.dumps(r, ensure_ascii=False) + "\n" for r in rows if r["split"] == "test"
    ).encode("utf-8")
    got = hashlib.sha256(test_bytes).hexdigest()
    want = STATS[lang]["sha256_test_frozen"]
    assert got == want, f"{lang}: frozen test set changed. Stop, results would be invalid."
    bad = {r["label"] for r in rows} - set(LABELS)
    assert not bad, f"{lang}: unknown labels {bad}"
    assert {r["language"] for r in rows} == {lang}, f"{lang}: mixed languages in file"
    data[lang] = {s: [r for r in rows if r["split"] == s] for s in ("train", "val", "test")}
    print(lang, {s: len(v) for s, v in data[lang].items()}, "| frozen test hash OK")

In [ ]:
# 5) Metric helpers (pure Python). Per language only; never pooled.
def softmax(row, T=1.0):
    m = max(row)
    e = [math.exp((x - m) / T) for x in row]
    s = sum(e)
    return [v / s for v in e]


def argmax_label(row):
    return LABELS[max(range(len(row)), key=row.__getitem__)]


def _counts(y_true, y_pred):
    tp, fp, fn = Counter(), Counter(), Counter()
    for t, p in zip(y_true, y_pred):
        if t == p:
            tp[t] += 1
        else:
            fp[p] += 1
            fn[t] += 1
    return tp, fp, fn


def per_class_f1(y_true, y_pred):
    tp, fp, fn = _counts(y_true, y_pred)
    out = {}
    for c in LABELS:
        prec = tp[c] / (tp[c] + fp[c]) if tp[c] + fp[c] else 0.0
        rec = tp[c] / (tp[c] + fn[c]) if tp[c] + fn[c] else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        out[c] = {"precision": prec, "recall": rec, "f1": f1, "support": tp[c] + fn[c]}
    return out


def macro_f1(y_true, y_pred):
    pc = per_class_f1(y_true, y_pred)
    return sum(v["f1"] for v in pc.values()) / len(pc)


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - m), min(1.0, c + m))


def bootstrap_macro_f1_ci(y_true, y_pred, n_boot=1000, seed=SEED):
    rng = random.Random(seed)
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = [rng.randrange(n) for _ in range(n)]
        scores.append(macro_f1([y_true[i] for i in idx], [y_pred[i] for i in idx]))
    scores.sort()
    return (scores[int(0.025 * n_boot)], scores[int(0.975 * n_boot) - 1])


def ece(confs, correct, bins=10):
    n = len(confs)
    total = 0.0
    for b in range(bins):
        lo, hi = b / bins, (b + 1) / bins
        idx = [i for i, c in enumerate(confs) if lo < c <= hi or (b == 0 and c == 0)]
        if idx:
            acc = sum(correct[i] for i in idx) / len(idx)
            conf = sum(confs[i] for i in idx) / len(idx)
            total += len(idx) / n * abs(acc - conf)
    return total


def evaluate(logits, rows, T=1.0):
    y_true = [r["label"] for r in rows]
    probs = [softmax(l, T) for l in logits]
    y_pred = [argmax_label(p) for p in probs]
    conf = [max(p) for p in probs]
    correct = [int(t == p) for t, p in zip(y_true, y_pred)]
    k, n = sum(correct), len(rows)
    return {
        "n": n,
        "accuracy": k / n,
        "accuracy_ci95": list(wilson_ci(k, n)),
        "macro_f1": macro_f1(y_true, y_pred),
        "macro_f1_ci95": list(bootstrap_macro_f1_ci(y_true, y_pred)),
        "ece": ece(conf, correct),
        "per_class": per_class_f1(y_true, y_pred),
    }


def report_markdown(report):
    out = [f"# Emotion model evaluation ({report['model_version']})", "",
           "Frozen test sets. Languages are reported separately and never pooled.", ""]
    for lang in LANGS:
        r = report["test"][lang]
        a_lo, a_hi = r["accuracy_ci95"]
        f_lo, f_hi = r["macro_f1_ci95"]
        out += [f"## {lang} (n = {r['n']})", "",
                f"- Accuracy: {r['accuracy']:.3f} (95% CI {a_lo:.3f}-{a_hi:.3f}, Wilson)",
                f"- Macro-F1: {r['macro_f1']:.3f} (95% CI {f_lo:.3f}-{f_hi:.3f}, bootstrap 1000)",
                f"- ECE (calibrated, 10 bins): {r['ece']:.3f}", "",
                "| label | precision | recall | F1 | support |", "|---|---|---|---|---|"]
        for c in LABELS:
            p = r["per_class"][c]
            out.append(f"| {c} | {p['precision']:.3f} | {p['recall']:.3f} | {p['f1']:.3f} | {p['support']} |")
        out.append("")
    out += [f"Temperature (fit on validation only): {report['temperature']}",
            f"Best epoch (validation macro-F1): {report['best']['epoch']}",
            f"Versions: {report['versions']}"]
    return "\n".join(out) + "\n"

In [ ]:
# 6) GPU, tokenizer, data loaders, class weights
import torch, transformers
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."
device = torch.device("cuda")
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
VERSIONS = {"torch": torch.__version__, "transformers": transformers.__version__,
            "gpu": torch.cuda.get_device_name(0)}
print(VERSIONS)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)


def collate(batch):
    enc = tok([r["text"] for r in batch], truncation=True, max_length=MAX_LEN,
              padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor([L2I[r["label"]] for r in batch])
    return enc


def make_dl(rows, shuffle=False, bs=128):
    return DataLoader(rows, batch_size=bs, shuffle=shuffle, collate_fn=collate, num_workers=2)


# Train on both languages together (one multilingual model);
# evaluation stays per language.
train_rows = data["en"]["train"] + data["roman_ur"]["train"]
train_dl = make_dl(train_rows, shuffle=True, bs=BATCH)
counts = Counter(r["label"] for r in train_rows)
weights = torch.tensor([len(train_rows) / (len(LABELS) * counts[l]) for l in LABELS],
                       dtype=torch.float, device=device)
print("train rows:", len(train_rows), dict(counts))
print("class weights:", {l: round(w, 3) for l, w in zip(LABELS, weights.tolist())})

In [ ]:
# 7) Train. The best epoch is chosen on VALIDATION only.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS),
    id2label=dict(enumerate(LABELS)), label2id=L2I).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_dl) * EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda")
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)


@torch.no_grad()
def torch_logits(m, rows):
    m.eval()
    out = []
    for b in make_dl(rows):
        b = {k: v.to(device) for k, v in b.items() if k != "labels"}
        with torch.autocast("cuda", dtype=torch.float16):
            out.extend(m(**b).logits.float().cpu().tolist())
    return out


best = {"score": -1.0}
for epoch in range(EPOCHS):
    model.train()
    t0, running = time.time(), 0.0
    for step, b in enumerate(train_dl):
        b = {k: v.to(device) for k, v in b.items()}
        labels = b.pop("labels")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**b).logits
        loss = loss_fn(logits.float(), labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        running += loss.item()
        if step % 200 == 0:
            print(f"epoch {epoch + 1} step {step}/{len(train_dl)} loss {running / (step + 1):.4f}")
    val_f1 = {}
    for lang in LANGS:
        rows = data[lang]["val"]
        preds = [argmax_label(l) for l in torch_logits(model, rows)]
        val_f1[lang] = round(macro_f1([r["label"] for r in rows], preds), 4)
    # Mean of per-language val F1 is used ONLY to pick the epoch, never reported as a headline.
    score = sum(val_f1.values()) / len(val_f1)
    print(f"epoch {epoch + 1}: {(time.time() - t0) / 60:.1f} min, val macro-F1 {val_f1}")
    if score > best["score"]:
        best = {"score": score, "epoch": epoch + 1, "val_macro_f1": val_f1}
        model.save_pretrained(BEST_DIR); tok.save_pretrained(BEST_DIR)
print("best:", best)

In [ ]:
# 8) Confidence calibration: temperature scaling fit on VALIDATION only
model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).to(device)
val_logits = {lang: torch_logits(model, data[lang]["val"]) for lang in LANGS}


def val_nll(T):
    tot, n = 0.0, 0
    for lang in LANGS:
        for l, r in zip(val_logits[lang], data[lang]["val"]):
            tot -= math.log(max(softmax(l, T)[L2I[r["label"]]], 1e-12))
            n += 1
    return tot / n


grid = [round(0.5 + 0.05 * i, 2) for i in range(91)]  # 0.50 .. 5.00
TEMPERATURE = min(grid, key=val_nll)
print("temperature:", TEMPERATURE, "| val NLL before", round(val_nll(1.0), 4),
      "after", round(val_nll(TEMPERATURE), 4))

In [ ]:
# 9) Export to ONNX, quantize to int8 (CPU inference in the app), sanity check
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

OUT_DIR.mkdir(parents=True, exist_ok=True)
cpu_model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).eval()


class LogitsOnly(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits


sample = tok(["export sample text"], return_tensors="pt")
args = (sample["input_ids"], sample["attention_mask"])
fp32_path = Path("/content/emotion_fp32.onnx")
kw = dict(input_names=["input_ids", "attention_mask"], output_names=["logits"],
          dynamic_axes={"input_ids": {0: "batch", 1: "seq"},
                        "attention_mask": {0: "batch", 1: "seq"},
                        "logits": {0: "batch"}},
          opset_version=17)
try:
    torch.onnx.export(LogitsOnly(cpu_model), args, str(fp32_path), dynamo=False, **kw)
except Exception as exc:  # newer torch may drop the legacy exporter
    print("legacy exporter failed, trying dynamo exporter:", exc)
    torch.onnx.export(LogitsOnly(cpu_model), args, str(fp32_path), dynamo=True, **kw)
onnx.checker.check_model(str(fp32_path))

int8_path = OUT_DIR / "emotion_int8.onnx"
quantize_dynamic(str(fp32_path), str(int8_path), weight_type=QuantType.QInt8)
tok.save_pretrained(OUT_DIR)
sess = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])


def onnx_logits(rows, bs=64):
    out = []
    for i in range(0, len(rows), bs):
        enc = tok([r["text"] for r in rows[i:i + bs]], truncation=True,
                  max_length=MAX_LEN, padding=True, return_tensors="np")
        feeds = {"input_ids": enc["input_ids"].astype("int64"),
                 "attention_mask": enc["attention_mask"].astype("int64")}
        out.extend(sess.run(["logits"], feeds)[0].tolist())
    return out


check_rows = data["en"]["val"][:150] + data["roman_ur"]["val"][:150]
agree = sum(argmax_label(a) == argmax_label(b)
            for a, b in zip(torch_logits(model, check_rows), onnx_logits(check_rows)))
print(f"int8 ONNX vs fp32 torch: same prediction on {agree}/{len(check_rows)} val rows")
print("int8 size (MB):", round(int8_path.stat().st_size / 1e6, 1))

In [ ]:
# 10) Final evaluation on the FROZEN test sets with the SHIPPED model (int8 ONNX)
MODEL_VERSION = "xlmr-base-emotion-" + time.strftime("%Y%m%d") + f"-T{TEMPERATURE}"
report = {"model_version": MODEL_VERSION, "model_name": MODEL_NAME, "labels": LABELS,
          "temperature": TEMPERATURE, "best": best, "versions": VERSIONS,
          "frozen_test_sha256": {l: STATS[l]["sha256_test_frozen"] for l in LANGS},
          "test": {}}
for lang in LANGS:
    t0 = time.time()
    report["test"][lang] = evaluate(onnx_logits(data[lang]["test"]), data[lang]["test"], TEMPERATURE)
    print(lang, "evaluated in", round((time.time() - t0) / 60, 1), "min")
md_report = report_markdown(report)
print(md_report)

In [ ]:
# 11) Package and download
import shutil
(OUT_DIR / "model_version.txt").write_text(MODEL_VERSION, encoding="utf-8")
(OUT_DIR / "calibration.json").write_text(
    json.dumps({"temperature": TEMPERATURE, "labels": LABELS, "max_len": MAX_LEN}, indent=2),
    encoding="utf-8")
(OUT_DIR / "eval_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
(OUT_DIR / "eval_report.md").write_text(md_report, encoding="utf-8")
zip_path = shutil.make_archive("/content/emotion_model", "zip", OUT_DIR)
print("files:", sorted(p.name for p in OUT_DIR.iterdir()))
from google.colab import files
files.download(zip_path)